In [1]:
import torch

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

TOKENIZER_REPO = "fracagnetta/tinystories.BPE8192"

tokenizer_path = hf_hub_download(
    repo_id=TOKENIZER_REPO,
    filename="tinystories.BPE8192.tokenizer.json",
    repo_type="dataset",
)

tokenizer = Tokenizer.from_file(tokenizer_path)

V = tokenizer.get_vocab_size()

print("vocab size:", V)

tinystories.BPE8192.tokenizer.json:   0%|          | 0.00/303k [00:00<?, ?B/s]

vocab size: 8192


In [15]:
text = "Once upon a time, there was a little girl named Lily."

encoded = tokenizer.encode(text)

print(encoded.tokens)
print(encoded.ids)

decoded = tokenizer.decode(encoded.ids)
print(decoded)

['Once', 'upon', 'a', 'time', ',', 'there', 'was', 'a', 'little', 'girl', 'named', 'Lily', '.']
[366, 371, 65, 331, 13, 338, 236, 65, 326, 408, 340, 316, 15]
Once upon a time , there was a little girl named Lily .


In [16]:
print("characters:", len(text))
print("tokens:", len(encoded.ids))
print("compression:", len(text) / len(encoded.ids))

characters: 53
tokens: 13
compression: 4.076923076923077


In [17]:
DATASET = "karpathy/tinystories-gpt4-clean"

train_stream = load_dataset(
    DATASET,
    split="train",
    streaming=True,
)

val_stream = load_dataset(
    DATASET,
    split="train",
    streaming=True,
)

train_texts = [
    row["text"]
    for row in train_stream.skip(20_000).take(20_000)
]

val_texts = [
    row["text"]
    for row in val_stream.skip(10_000).take(1_000)
]

print("train:", len(train_texts))
print("val:", len(val_texts))

train: 20000
val: 1000


In [18]:
story = train_texts[0]

enc = tokenizer.encode(story)

print(story[:500])
print()
print("num chars :", len(story))
print("num tokens:", len(enc.ids))
print()
print(enc.tokens[:50])
print(enc.ids[:50])

Once upon a time, in a small house, there lived a clumsy dog named Spot. Spot liked to play and run all day. In the evening, when the sun went down, Spot would get very tired.
One evening, after a long day of playing, Spot was very thirsty. He wanted to drink some water. So, he went to his bowl to drink. But, because he was so clumsy, he knocked the bowl over and water went everywhere!
Spot felt bad about being clumsy, but his owner, a little girl named Sarah, just laughed. She said, "Silly Spot

num chars : 715
num tokens: 169

['Once', 'upon', 'a', 'time', ',', 'in', 'a', 'small', 'house', ',', 'there', 'lived', 'a', 'clumsy', 'dog', 'named', 'Spot', '.', 'Spot', 'liked', 'to', 'play', 'and', 'run', 'all', 'day', '.', 'In', 'the', 'evening', ',', 'when', 'the', 'sun', 'went', 'down', ',', 'Spot', 'would', 'get', 'very', 'tired', '.', 'One', 'evening', ',', 'after', 'a', 'long', 'day']
[366, 371, 65, 331, 13, 228, 65, 472, 510, 13, 338, 531, 65, 2549, 395, 340, 483, 15, 483, 453, 226,

In [19]:
eos_id = tokenizer.token_to_id("<eos>")

print("eos_id:", eos_id)

eos_id: 1


In [20]:
def tokenize_stories(texts, tokenizer, eos_id=None):
    tokenized = []

    for text in texts:
        ids = tokenizer.encode(text).ids

        if eos_id is not None:
            ids.append(eos_id)

        tokenized.append(
            torch.tensor(ids, dtype=torch.long)
        )

    return tokenized


train_tokens = tokenize_stories(
    train_texts,
    tokenizer,
    eos_id=eos_id,
)

val_tokens = tokenize_stories(
    val_texts,
    tokenizer,
    eos_id=eos_id,
)

print("train stories:", len(train_tokens))
print("val stories:", len(val_tokens))

print("first tokenized story shape:", train_tokens[0].shape)

train stories: 20000
val stories: 1000
first tokenized story shape: torch.Size([170])


In [21]:
def sample_batch(tokenized_stories, batch_size, context_length, device):
    valid = [
        tokens
        for tokens in tokenized_stories
        if len(tokens) >= context_length + 1
    ]

    xs = []
    ys = []

    for _ in range(batch_size):
        story = valid[
            torch.randint(0, len(valid), (1,)).item()
        ]

        max_start = len(story) - context_length

        start = torch.randint(
            0,
            max_start,
            (1,)
        ).item()

        chunk = story[
            start : start + context_length + 1
        ]

        x = chunk[:-1]
        y = chunk[1:]

        xs.append(x)
        ys.append(y)

    x = torch.stack(xs).to(device)
    y = torch.stack(ys).to(device)

    return x, y

In [24]:
batch_size = 16
context_length = 128

device = 'cpu'

In [25]:
x_batch, y_batch = sample_batch(
    train_tokens,
    batch_size=16,
    context_length=128,
    device=device,
)

print(x_batch.shape)
print(y_batch.shape)

torch.Size([16, 128])
torch.Size([16, 128])


In [26]:
print(tokenizer.decode(x_batch[0].cpu().tolist()))
print("--- TARGET ---")
print(tokenizer.decode(y_batch[0].cpu().tolist()))

with a sticker , so we know it ' s ours ," Ben said . He took a sticker from his pocket and stuck it on the rock . They kept playing until they saw a dog running towards them . The dog was big and loud . It barked and jumped on their cart . It knocked over their things and took the shiny rock in its mouth . " Hey , stop , that ' s our rock !" Ben shouted . He tried to get the rock back , but the dog was too strong . Lily was scared . She ran to a bench where a lady was sitting . The lady saw what was happening and called the dog . " Rex ,
--- TARGET ---
a sticker , so we know it ' s ours ," Ben said . He took a sticker from his pocket and stuck it on the rock . They kept playing until they saw a dog running towards them . The dog was big and loud . It barked and jumped on their cart . It knocked over their things and took the shiny rock in its mouth . " Hey , stop , that ' s our rock !" Ben shouted . He tried to get the rock back , but the dog was too strong . Lily was scared . She ran

In [29]:
from transformers_from_scratch.model import TinyDecoderLM

In [31]:
V = tokenizer.get_vocab_size()

d_model = 128
n_heads = 4
d_ff = 512
n_layers = 4

model = TinyDecoderLM(
    vocab_size=V,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    n_layers=n_layers,
).to(device)

num_params = sum(p.numel() for p in model.parameters())

print("V:", V)
print("params:", f"{num_params:,}")

V: 8192
params: 2,888,448


In [32]:
x_fixed, y_fixed = sample_batch(
    train_tokens,
    batch_size=8,
    context_length=128,
    device=device,
)

print(x_fixed.shape)
print(y_fixed.shape)

torch.Size([8, 128])
torch.Size([8, 128])


In [33]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.0,
)

In [35]:
import torch.nn.functional as F

model.train()

for step in range(1000):
    logits = model(x_fixed)                     # (B, n, V)

    loss = F.cross_entropy(
        logits.reshape(-1, V),                 # (B*n, V)
        y_fixed.reshape(-1),                   # (B*n,)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())

0 9.17960262298584
50 1.1465944051742554
100 0.04425456002354622
150 0.018735690042376518
200 0.012088502757251263
250 0.008856210857629776
300 0.00697295879945159
350 0.005759917665272951
400 0.004918565973639488
450 0.004308915231376886
500 0.003849797649309039
550 0.0034954838920384645
600 0.0032121867407113314
650 0.0029837763868272305
700 0.002796049928292632
750 0.002644012216478586
800 0.0025084514636546373
850 0.0023960426915436983
900 0.0022996654734015465
950 0.0022255738731473684


In [38]:
import numpy as np
np.log(8192)

np.float64(9.010913347279288)

In [39]:
prompt = x_fixed[0, :20].unsqueeze(0)

In [40]:
prompt

tensor([[ 375, 1144,  608,  230,  236,  289,  286,  227, 1331,   15,  322,  267,
           13,  271,  886,   65,  393,  298,  340,  418]])

In [41]:
model.eval()

tokens = prompt.clone()

with torch.no_grad():
    for _ in range(60):
        logits = model(tokens)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tokens = torch.cat([tokens, next_token], dim=1)

print("GENERATED:")
print(tokenizer.decode(tokens[0].cpu().tolist()))

print("\nTRAINING EXAMPLE:")
print(tokenizer.decode(x_fixed[0].cpu().tolist()))

GENERATED:
toy cars because it was very big and colorful . One day , Tim met a new friend named Sue . Sue had a toy car too , but her car was small and not as colorful as Tim ' s car . They decided to play together with their toy cars . Tim said , " Let ' s push our cars and see whose car can go the farthest !" Sue agreed and said , " Okay ,

TRAINING EXAMPLE:
toy cars because it was very big and colorful . One day , Tim met a new friend named Sue . Sue had a toy car too , but her car was small and not as colorful as Tim ' s car . They decided to play together with their toy cars . Tim said , " Let ' s push our cars and see whose car can go the farthest !" Sue agreed and said , " Okay , let ' s do it !" They both pushed their cars with all their might . Tim ' s car went a few inches farther than Sue ' s car . They laughed and clapped their hands with joy . They played with their cars all day long


In [43]:
torch.manual_seed(0)

model = TinyDecoderLM(
    vocab_size=V,
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_layers=4,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)

In [44]:
def estimate_val_loss(
    model,
    val_tokens,
    batch_size,
    context_length,
    device,
    n_batches=20,
):
    model.eval()

    losses = []

    with torch.no_grad():
        for _ in range(n_batches):
            x, y = sample_batch(
                val_tokens,
                batch_size,
                context_length,
                device,
            )

            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, V),
                y.reshape(-1),
            )

            losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)

In [45]:
def grad_norm(model):
    total_sq = 0.0

    for p in model.parameters():
        if p.grad is not None:
            total_sq += p.grad.detach().pow(2).sum().item()

    return total_sq ** 0.5

In [46]:
num_steps = 2000
eval_every = 100

train_history = []
val_history = []
grad_history = []
lr_history = []

model.train()

for step in range(num_steps):
    x, y = sample_batch(
        train_tokens,
        batch_size=batch_size,
        context_length=context_length,
        device=device,
    )

    logits = model(x)

    loss = F.cross_entropy(
        logits.reshape(-1, V),
        y.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()

    gnorm = grad_norm(model)

    optimizer.step()

    lr = optimizer.param_groups[0]["lr"]

    train_history.append(loss.item())
    grad_history.append(gnorm)
    lr_history.append(lr)

    if step % eval_every == 0:
        val_loss = estimate_val_loss(
            model,
            val_tokens,
            batch_size=batch_size,
            context_length=context_length,
            device=device,
            n_batches=20,
        )

        val_history.append((step, val_loss))

        print(
            f"step={step:4d} "
            f"train={loss.item():.4f} "
            f"val={val_loss:.4f} "
            f"grad={gnorm:.3f} "
            f"lr={lr:.2e}"
        )

step=   0 train=9.1885 val=9.0908 grad=0.781 lr=3.00e-04
step= 100 train=5.3212 val=5.3698 grad=0.537 lr=3.00e-04
step= 200 train=4.5758 val=4.6945 grad=0.443 lr=3.00e-04
step= 300 train=4.4254 val=4.3179 grad=0.474 lr=3.00e-04
step= 400 train=4.2874 val=4.0293 grad=0.583 lr=3.00e-04
step= 500 train=3.6610 val=3.8557 grad=0.664 lr=3.00e-04
step= 600 train=3.7682 val=3.7445 grad=0.649 lr=3.00e-04
step= 700 train=3.5525 val=3.6624 grad=0.648 lr=3.00e-04
step= 800 train=3.6485 val=3.5746 grad=0.671 lr=3.00e-04
step= 900 train=3.6291 val=3.5402 grad=0.846 lr=3.00e-04
step=1000 train=3.4555 val=3.4732 grad=0.761 lr=3.00e-04
step=1100 train=3.2976 val=3.4721 grad=0.726 lr=3.00e-04
step=1200 train=3.3090 val=3.3766 grad=0.829 lr=3.00e-04
step=1300 train=3.2176 val=3.3162 grad=0.777 lr=3.00e-04
step=1400 train=3.2716 val=3.2785 grad=0.841 lr=3.00e-04
step=1500 train=3.0005 val=3.2184 grad=0.773 lr=3.00e-04
step=1600 train=3.0112 val=3.2413 grad=0.823 lr=3.00e-04
step=1700 train=3.0834 val=3.17